# 02 Embedding + MLP 문자 모델

## 목적
`block_size=3`으로 세 글자 문맥을 만들고, token embedding, flatten, MLP hidden layer를 거쳐 다음 문자를 예측합니다.

In [1]:
from pathlib import Path
import sys, torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tokenizer import CharTokenizer, load_text
from src.baselines import MLPNextChar

torch.manual_seed(42)
text = load_text(str(ROOT / 'data/korean_finance_corpus.txt'))
tokenizer = CharTokenizer(text)
ids = tokenizer.encode(text[:160])
print('vocab_size:', tokenizer.vocab_size)

vocab_size: 370


## 핵심 코드
슬라이딩 윈도우로 `x=[t0,t1,t2]`, `y=t3` 형태의 학습 예제를 만듭니다. 임베딩 shape은 `(B, 3, C)`, flatten shape은 `(B, 3*C)`입니다.

In [2]:
block_size = 3
xs, ys = [], []
for i in range(48):
    xs.append(ids[i:i+block_size])
    ys.append(ids[i+block_size])
x = torch.tensor(xs, dtype=torch.long)
y = torch.tensor(ys, dtype=torch.long)

model = MLPNextChar(tokenizer.vocab_size, block_size=3, n_embd=12, hidden=48)
logits, loss, emb, flat = model(x, y)
print('x shape:', tuple(x.shape))
print('embedding shape:', tuple(emb.shape))
print('flatten shape:', tuple(flat.shape))
print('logits shape:', tuple(logits.shape))
print('loss:', round(float(loss), 4))

x shape: (48, 3)
embedding shape: (48, 3, 12)
flatten shape: (48, 36)
logits shape: (48, 370)
loss: 5.9619


/tmp/ipykernel_68453/2540910069.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  print('loss:', round(float(loss), 4))


In [3]:
opt = torch.optim.AdamW(model.parameters(), lr=0.03)
for _ in range(10):
    _, loss, _, _ = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
context = tokenizer.encode('인공지')[-3:]
with torch.no_grad():
    pred, _, _, _ = model(torch.tensor([context]))
next_id = int(torch.argmax(pred[0]).item())
print('context:', tokenizer.decode(context))
print('predicted next char:', tokenizer.decode([next_id]))

context: 인공지
predicted next char: 능


## 이 단계의 한계
MLP는 항상 고정된 세 글자만 봅니다. 더 긴 문맥이나 위치별 출력은 처리하지 못합니다.

## 다음 단계가 해결하는 점
다음 노트북은 같은 구조를 더 긴 한국어 말뭉치에 적용하고, 데이터 도메인이 바뀌면 학습 예제와 한계가 어떻게 보이는지 확인합니다.